# Step 02 — ALKIS / LoD2 extraction

Assemble the raw ALKIS building dataset from LGLN's LoD2 open data for
Niedersachsen: select the region's tiles, download them, and merge them into
one layer.

| | |
|---|---|
| **Reads** | `data/input/lgln-opengeodata-lod2.geojson`, `data/input/regionalverband_area.gpkg` |
| **Writes** | `02_lod2_region_tiles.gpkg` · `02_alkis_lod2_raw.gpkg` |
| **Needs** | `ogr2ogr` (system binary), `requests` |
| **Runtime** | ~16 min cold, ~14 min warm — the merge always reruns |
| **Disk** | 0.44 GB cached zips + 4.26 GB output |
| **Result** | 1,385,279 buildings as 4,891,343 surface rows |

This is the equivalent of `ALKIS_LOD2-data_extraction.ipynb` in the original
pipeline and does the same job and nothing else: **filter the tile index,
download, merge.**

The original's separate unzip stage is gone — GDAL reads a shapefile straight
out of its `.zip` through `/vsizip/`, which produces a byte-identical merge
(verified on three tiles: same row count, same `gml_id` count, same column set,
same total area, same file size) while never extracting anything. The zips
inflate ~25x, so this avoids roughly **11 GB** of shapefiles and keeps peak disk
at 4.7 GB.

Measured on the full region: 3,620 tiles downloaded in 67 s, merged in 849 s,
`02_alkis_lod2_raw.gpkg` = 4.26 GB. The original pipeline's equivalent
`merged_all.gpkg` was 5.09 GB; the difference is `-dim XY` dropping the Z
ordinate.

**`02_alkis_lod2_raw.gpkg` is raw.** One row per LoD2 *surface*, not per
building — a tile carries ground, wall and roof surfaces as separate rows
sharing one `gml_id`. Region-wide that is **4,891,343 rows for 1,385,279
buildings, 3.53 surfaces each**. So `gml_id` is **not unique** in this file,
`area_m2` is not computed, and no volume exists yet.
Reducing surfaces to buildings, computing volume, choosing a size threshold,
labelling the `function` codes and deciding whether to merge adjacent polygons
are all later steps' decisions. Section 5 measures the file so those decisions
have numbers to work from.

The download skips what is already on disk, so an interrupted run resumes
rather than restarting.

In [ ]:
import os, shutil, sys
from pathlib import Path

# --- locate the pipeline root -------------------------------------------------
# Same reason as step 01: a notebook's working directory is not necessarily its
# own folder, so Path('..') is unreliable. Find the root by its marker file.
def _find_root(start):
    for d in (start, *start.parents):
        if (d / 'config.py').is_file() and (d / 'lib' / 'checks.py').is_file():
            return d
    return None

_nb_dir = Path(globals()['__vsc_ipynb_file__']).parent if '__vsc_ipynb_file__' in globals() else None
ROOT_DIR = _find_root(_nb_dir) if _nb_dir else None
ROOT_DIR = ROOT_DIR or _find_root(Path.cwd())
if ROOT_DIR is None:
    raise RuntimeError(
        'Cannot find the pipeline root (the folder containing config.py). '
        f'Looked upward from notebook dir {_nb_dir} and cwd {Path.cwd()}.'
    )
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# --- point GDAL/PROJ at this env's data files ---------------------------------
# Must run before geopandas is imported. ogr2ogr inherits these too: without
# GDAL_DATA it prints `Cannot find tms_NZTM2000.json` on every single call,
# which across 3,620 tiles buries the real output.
_share = Path(sys.prefix) / 'Library' / 'share'
if not _share.is_dir():
    _share = Path(sys.prefix) / 'share'
if (_share / 'gdal').is_dir():
    os.environ.setdefault('GDAL_DATA', str(_share / 'gdal'))
if (_share / 'proj').is_dir():
    os.environ.setdefault('PROJ_LIB', str(_share / 'proj'))

import subprocess, zipfile, time
from concurrent.futures import ThreadPoolExecutor

# --- find a working ogr2ogr ---------------------------------------------------
# Verified by running it, for the two reasons step 01 verifies osmium: a kernel
# started without `conda activate` has none of this env's binaries on PATH, and
# shutil.which returns the PATHEXT-upper-cased name on Windows.
for _bin in (Path(sys.prefix) / 'Library' / 'bin',
             Path(sys.prefix) / 'Scripts',
             Path(sys.prefix) / 'bin'):
    if _bin.is_dir() and str(_bin) not in os.environ.get('PATH', ''):
        os.environ['PATH'] = str(_bin) + os.pathsep + os.environ.get('PATH', '')


def _find_tool(stem):
    cands = []
    for d in (Path(sys.prefix) / 'Library' / 'bin',
              Path(sys.prefix) / 'Scripts',
              Path(sys.prefix) / 'bin'):
        cands += [d / f'{stem}.exe', d / stem]
    found = shutil.which(stem)
    if found:
        f = Path(found)
        cands += [f.with_suffix(f.suffix.lower()), f]
    for c in cands:
        if not c.is_file():
            continue
        try:
            r = subprocess.run([str(c), '--version'], capture_output=True, text=True)
        except OSError:
            continue
        if r.returncode == 0:
            return str(c), r.stdout.splitlines()[0]
    return None, None


OGR2OGR, _gdal_version = _find_tool('ogr2ogr')
if OGR2OGR is None:
    raise RuntimeError(
        'No working ogr2ogr found. Section 4 needs it: LoD2 tiles are MultiPatch '
        'shapefiles, which shapely cannot parse (`Unknown WKB type 16`), so GDAL '
        'has to do the conversion. Install it with:  '
        'conda install -c conda-forge gdal'
    )

import requests
import pandas as pd
import geopandas as gpd

from config import (
    STUDY_BOUNDARY_FILE, TARGET_CRS, OUTPUT_DIR,
    LOD2_TILE_INDEX_FILE, LOD2_CACHE_DIR, LOD2_ZIP_DIR,
    LOD2_REGION_TILES_FILE, ALKIS_RAW_FILE,
    LOD2_DOWNLOAD_WORKERS, LOD2_DOWNLOAD_TIMEOUT_S, LOD2_DOWNLOAD_RETRIES,
    LOD2_MERGE_GEOM_TYPE, LOD2_MERGE_FLATTEN_Z,
)
from lib.checks import require_file, require_non_empty, require_crs, require_unique

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOD2_ZIP_DIR.mkdir(parents=True, exist_ok=True)

print('Root       :', ROOT_DIR)
print('ogr2ogr    :', _gdal_version, '|', OGR2OGR)
print('Target CRS :', TARGET_CRS)
print('Cache      :', LOD2_CACHE_DIR)

## 1. Input contract

Both inputs are checked before a single tile is fetched — a missing boundary
should fail in a second, not an hour into the downloads.

In [ ]:
require_file(LOD2_TILE_INDEX_FILE, 'LoD2 tile index')
require_file(STUDY_BOUNDARY_FILE, 'study boundary')

boundary = gpd.read_file(STUDY_BOUNDARY_FILE)
require_non_empty(boundary, 'boundary')
print(f'  ..  boundary: {boundary.crs.to_string()}, '
      f'{boundary.to_crs(TARGET_CRS).geometry.area.sum() / 1e6:,.0f} km2')

## 2. Select the region's tiles

The index is the statewide catalogue: 37,928 tiles, each a polygon plus a `shp`
URL pointing at a zipped shapefile. It is EPSG:4326 whatever the tiles
themselves use, so the boundary is reprojected to *it* rather than the reverse.

`intersects`, not `within`: a tile the region border cuts through still holds
buildings inside the region. Tiles are 1 km squares, so the overhang is bounded
and harmless — trimming it is a later step's business, if it matters at all.

The selection is written out so the download set is auditable in QGIS, and so a
missing tile can be traced back to a specific square.

In [ ]:
tiles_all = gpd.read_file(LOD2_TILE_INDEX_FILE)
require_crs(tiles_all, 'EPSG:4326', 'tile index')
print(f'  ..  statewide tiles: {len(tiles_all):,}')

boundary_4326 = boundary.to_crs(epsg=4326).geometry.union_all()
tiles = tiles_all[tiles_all.geometry.intersects(boundary_4326)].copy().reset_index(drop=True)
require_non_empty(tiles, 'region tiles')

# One row per tile is asserted, not assumed: a duplicated `shp` URL would be
# downloaded twice and merged twice.
require_unique(tiles, 'shp', 'region tiles')

bad = tiles[~tiles['shp'].astype(str).str.startswith('http')]
if len(bad):
    raise AssertionError(f'{len(bad)} region tiles carry no usable `shp` URL')

tiles['tile_name'] = (tiles['shp'].astype(str)
                      .str.rsplit('/', n=1).str[-1]
                      .str.replace('.zip', '', regex=False))
require_unique(tiles, 'tile_name', 'region tiles')

print(f'  ..  kept {len(tiles):,} of {len(tiles_all):,} '
      f'({100 * len(tiles) / len(tiles_all):.1f} %)')
print(f"  ..  currency (Aktualitaet): {tiles['Aktualitaet'].min()} .. "
      f"{tiles['Aktualitaet'].max()}")

if LOD2_REGION_TILES_FILE.exists():
    LOD2_REGION_TILES_FILE.unlink()
tiles.to_file(LOD2_REGION_TILES_FILE, layer='tiles', driver='GPKG')
print(f'  ok  tile selection -> {LOD2_REGION_TILES_FILE.name}')

## 3. Download the tiles

One zipped shapefile per tile from LGLN's object store, `LOD2_DOWNLOAD_WORKERS`
at a time. The original pipeline fetched these one at a time under `tqdm`; the
only change here is concurrency plus the three safeguards below.

* An existing zip is **verified, not assumed** — `testzip()` catches the
  truncated file an interrupted kernel leaves behind, and it is re-fetched.
  Skipping on mere existence, as the original did, silently keeps a corrupt tile.
* A download lands on a `.part` file and is renamed only once complete, so a
  `.zip` on disk is always a whole archive.
* Failures are collected and reported, not raised. A handful of bad tiles must
  not discard an hour of good ones — rerun the cell and only those retry.

In [ ]:
def _zip_ok(path):
    # A complete, readable archive that actually contains a shapefile.
    try:
        with zipfile.ZipFile(path) as z:
            if z.testzip() is not None:
                return False
            return any(n.lower().endswith('.shp') for n in z.namelist())
    except (zipfile.BadZipFile, OSError):
        return False


def _fetch(job):
    url, name = job
    dest = LOD2_ZIP_DIR / f'{name}.zip'
    if dest.exists():
        if _zip_ok(dest):
            return ('cached', name, None)
        dest.unlink()          # truncated by an interrupted run

    part = dest.with_suffix('.part')
    last = None
    for attempt in range(LOD2_DOWNLOAD_RETRIES):
        try:
            with requests.get(url, stream=True, timeout=LOD2_DOWNLOAD_TIMEOUT_S) as r:
                r.raise_for_status()
                with open(part, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=1 << 16):
                        if chunk:
                            f.write(chunk)
            if not _zip_ok(part):
                raise OSError('downloaded file is not a readable zip')
            part.replace(dest)  # atomic rename: no half-written .zip is visible
            return ('downloaded', name, None)
        except Exception as e:
            last = e
            time.sleep(1.5 * (attempt + 1))
    part.unlink(missing_ok=True)
    return ('failed', name, f'{type(last).__name__}: {last}')


jobs = list(zip(tiles['shp'].astype(str), tiles['tile_name']))
print(f'Downloading {len(jobs):,} tiles, {LOD2_DOWNLOAD_WORKERS} at a time '
      f'(cold: ~70 s for this region; fully cached: seconds) ...', flush=True)

t0 = time.perf_counter()
results = []
with ThreadPoolExecutor(max_workers=LOD2_DOWNLOAD_WORKERS) as pool:
    for i, res in enumerate(pool.map(_fetch, jobs), 1):
        results.append(res)
        if i % 250 == 0 or i == len(jobs):
            ok = sum(1 for s, _, _ in results if s != 'failed')
            print(f'  ..  {i:>5,}/{len(jobs):,}  ok={ok:,}  '
                  f'[{time.perf_counter() - t0:,.0f}s]', flush=True)

print()
for k, v in pd.Series([s for s, _, _ in results]).value_counts().items():
    print(f'  ..  {k:<11} {v:>6,}')

failed = [(n, e) for s, n, e in results if s == 'failed']
if failed:
    print(f'\n  !!  {len(failed)} tiles failed. Rerun this cell to retry just these:')
    for n, e in failed[:10]:
        print(f'        {n}: {e}')
    if len(failed) > 10:
        print(f'        ... and {len(failed) - 10} more')

zips = sorted(LOD2_ZIP_DIR.glob('*.zip'))
print(f'\n  ok  {len(zips):,} zips on disk, '
      f'{sum(p.stat().st_size for p in zips) / 1e9:,.2f} GB')

## 4. Merge into one layer

`ogr2ogr -append`, once per tile, into a single `buildings` layer — the same
approach as the original pipeline's final cell, which built `merged_all.gpkg`
that way.

**Why GDAL and not geopandas.** LoD2 shapefiles hold **MultiPatch** geometry —
3D surfaces. `geopandas.read_file` on a raw tile dies with
`GEOSException: ParseException: Unknown WKB type 16`, because shapely has no TIN
type. GDAL is the only thing here that can read them, which is why the merge is
a subprocess and not a `pd.concat`.

**Read straight from the zip.** The source path is `/vsizip/<tile>.zip`, GDAL's
virtual filesystem, so nothing is extracted to disk. Pointing it at the archive
rather than an inner `.shp` also means a tile shipping more than one shapefile
contributes all of them, which is what the original's recursive `*.shp` glob
did. Verified against extract-then-merge on three tiles: identical row count,
`gml_id` count, column set and total area, byte-for-byte the same file size.

One flag the original did not pass: `-nlt MULTIPOLYGONZ`. Appending 3,620 tiles
into one layer needs a single declared geometry type, or a tile whose surfaces
come back as plain `POLYGON` is rejected by a layer created as multi.

**The Z ordinate is kept, and that is not free — it costs ~0.8 GB.** An earlier
version of this notebook passed `-dim XY` to save exactly that, reasoning that
the heights are already in `measHeight`/`Firsthoehe`/`Traufhoehe`/`AbsHoehe`.
That reasoning was wrong. Z does not carry the heights; it carries the **surface
semantics**. It is the only thing that separates a ground surface from a roof
surface — no attribute column does, because every row of a building repeats
identical attribute values.

Measured on a 60-tile sweep: classifying surfaces by Z yields exactly one
`GROUND` surface for **13,833 of 13,833 parts (100.00 %)**, holding across
46.8–824.9 m of terrain, both `DqDach` groups and every surface count. Discard Z
and the only fallback is the original pipeline's "largest area per `gml_id`",
which picks the roof instead of the footprint whenever the roof overhangs —
wrong for **1.15 % of parts, median +32 %, worst +255 %**. Z also makes an exact
volume computable instead of `area × ridge height`, which overstates
pitched-roof buildings by 20–46 %.

The first tile **creates** the layer and every later one appends to it, so a
partial merge cannot be resumed: the output is deleted first and the whole merge
redone, every run. That is deliberate — appending to a layer of unknown state is
how you get silent duplicates — and it is why a warm rerun still costs ~14 min.
Caching protects the downloads, but at 67 s those turned out to be the cheap part:
the merge is 92 % of the runtime.

In [ ]:
if ALKIS_RAW_FILE.exists():
    try:
        ALKIS_RAW_FILE.unlink()
    except PermissionError as e:
        raise RuntimeError(
            f'{ALKIS_RAW_FILE.name} is locked by another process, so it cannot be '
            'replaced. QGIS holds a GeoPackage open for as long as the layer is '
            'loaded - remove the layer (or close the project) and run this cell '
            f'again. Original error: {e}'
        ) from None

_base = [OGR2OGR, '-f', 'GPKG', '-nln', 'buildings',
         '-nlt', LOD2_MERGE_GEOM_TYPE, '-t_srs', TARGET_CRS]
if LOD2_MERGE_FLATTEN_Z:
    _base += ['-dim', 'XY']

print(f'Merging {len(zips):,} tiles into {ALKIS_RAW_FILE.name} '
      f'(expect ~14 min; it slows as the GeoPackage grows) ...', flush=True)

t0 = time.perf_counter()
merge_failed = []
for i, z in enumerate(zips, 1):
    # GDAL's virtual filesystem: read the shapefile inside the archive, with
    # nothing extracted. as_posix() because /vsizip/ paths use forward slashes
    # even on Windows.
    src = '/vsizip/' + z.as_posix()
    # -update -append only after the first tile, which creates the layer.
    extra = [] if i == 1 else ['-update', '-append']
    r = subprocess.run(_base + extra + [str(ALKIS_RAW_FILE), src],
                       capture_output=True, text=True)
    if r.returncode != 0:
        merge_failed.append((z.stem, (r.stderr or r.stdout).strip()[:160]))
    if i % 200 == 0 or i == len(zips):
        mb = ALKIS_RAW_FILE.stat().st_size / 1e6 if ALKIS_RAW_FILE.exists() else 0
        print(f'  ..  {i:>5,}/{len(zips):,}  {mb:>9,.0f} MB  '
              f'[{time.perf_counter() - t0:,.0f}s]', flush=True)

if merge_failed:
    print(f'\n  !!  {len(merge_failed)} tiles failed to merge:')
    for n, e in merge_failed[:10]:
        print(f'        {n}: {e}')
    if len(merge_failed) > 10:
        print(f'        ... and {len(merge_failed) - 10} more')

print(f'\n  ok  {ALKIS_RAW_FILE.name}: '
      f'{ALKIS_RAW_FILE.stat().st_size / 1e9:,.2f} GB  '
      f'[{time.perf_counter() - t0:,.0f}s]')

## 5. Verify and measure

The merged layer is read back through GDAL's own metadata rather than loaded
into memory — it is a multi-gigabyte file and a full read would be pointless
here.

What the numbers mean, and why `gml_id` is deliberately not checked for
uniqueness: a LoD2 tile stores one row **per surface class**, all sharing a
`gml_id` and repeating the same attributes. For `DENILD61000068Eu` there are
three rows — the two roof planes (186.9 m² projected), the six walls (0.0 m²
once flattened, being vertical), and the ground surface (186.9 m², the actual
footprint).

Region-wide that comes to **4,891,343 surface rows for 1,385,279 buildings,
3.53 each**, median 4 and max 4 — but **min 1**, so a minority of buildings
arrive as a single surface. Any reduction rule has to handle that case rather
than assume a ground surface is always present.

Reducing these surfaces to one row per building is the first thing the next step
has to decide. Rerunning this cell after a re-download is also how you check
whether the region's structure has changed under you.

In [ ]:
import pyogrio

info = pyogrio.read_info(ALKIS_RAW_FILE, layer='buildings')
print(f'  ..  layer      : buildings')
print(f'  ..  rows       : {info["features"]:,}')
print(f'  ..  geometry   : {info["geometry_type"]}')
print(f'  ..  CRS        : {info["crs"]}')
print(f'  ..  fields     : {len(info["fields"])}')
print(f'  ..  bounds     : {tuple(round(v) for v in info["total_bounds"])}')
print()
print('  ..  columns:', ', '.join(info['fields']))

# gml_id alone, no geometry: cheap enough on a file this size and it is the one
# number the next step needs.
ids = pyogrio.read_dataframe(ALKIS_RAW_FILE, layer='buildings',
                             columns=['gml_id'], read_geometry=False)
n_bld = ids['gml_id'].nunique()
print()
print(f'  ..  surface rows        : {len(ids):,}')
print(f'  ..  distinct gml_id     : {n_bld:,}')
print(f'  ..  surfaces / building : {len(ids) / n_bld:.2f}')
print(f'  ..  rows with no gml_id : {int(ids["gml_id"].isna().sum()):,}')

per = ids['gml_id'].value_counts()
print(f'  ..  surfaces per building: min {per.min()}, median {per.median():.0f}, '
      f'max {per.max():,}')

require_non_empty(ids, 'merged surfaces')
print(f'\n  ok  extraction complete: {n_bld:,} buildings as {len(ids):,} surface rows')

## 6. QGIS checkpoint

Load `02_alkis_lod2_raw.gpkg` and `02_lod2_region_tiles.gpkg` alongside
`regionalverband_area.gpkg`, then confirm:

1. **Coverage** — the buildings blanket the whole region with no empty 1 km
   squares. A square hole is a tile that failed to download or merge; the
   section 3 and 4 reports name it.
2. **Alignment** — footprints sit on a basemap correctly and line up with
   `01_all_buildings_osm.gpkg`. Everything in the Atlantic near (0, 0) means a
   CRS was assigned rather than reprojected.
3. **Overlapping surfaces are expected** — clicking one building returns
   several features with the same `gml_id`. That is the raw structure, not an
   error. Filter `"gml_id" = '<some id>'` and you should see the ground surface
   and the roof planes stacked on each other, plus wall rows with no visible
   area.

Note that this layer is not the deliverable — it is the source for the next
step, which reduces surfaces to buildings and computes volume.